# CQI RDD Analysis

In [1]:
%%configure -f
{
    "conf": {
        "spark.pyspark.python": "python3",
        "spark.pyspark.virtualenv.enabled": "true",
        "spark.pyspark.virtualenv.type": "native",
        "spark.pyspark.virtualenv.bin.path": "/usr/bin/virtualenv"
    }
}


## 1. Setting

In [3]:
# Main run configuration
# On EMR, use S3. For local debugging, change DATA_SOURCE to "local".
DATA_SOURCE = "s3"  # "s3" for EMR, "local" for local debugging
SAMPLE_MONTH = "2022-11"
CUTOFF_DATE = "2022-12-01"

# Local paths, used only when DATA_SOURCE = "local" or for local notebook outputs.
LOCAL_POSTS_DIR = "posts_cqi_with_tier"
LOCAL_OUTPUT_DIR = "outputs"

# S3 paths for AWS Academy / EMR.
S3_POSTS_PATH = "s3://macs30123-final-pj-yanjing-li-2026/posts_cqi_with_tier"
S3_OUTPUT_DIR = "s3://macs30123-final-pj-yanjing-li-2026/outputs"

# RDD debug group.
DEBUG_TIER = "novice"
DEBUG_TAG_CATEGORY = "data_science"


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [43]:
from pathlib import Path
import glob
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import os


warnings.filterwarnings("ignore")
plt.style.use("default")

def show_table(obj, n=20):
    """Notebook display fallback for EMR/Livy kernels."""
    try:
        display(obj.head(n) if hasattr(obj, "head") else obj)
    except NameError:
        if hasattr(obj, "head"):
            print(obj.head(n).to_string(index=False))
        else:
            print(obj)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


LOCAL_WORK_DIR = PROJECT_ROOT
if str(LOCAL_WORK_DIR) == "/" or not os.access(str(LOCAL_WORK_DIR), os.W_OK):
    LOCAL_WORK_DIR = Path.home()

if DATA_SOURCE == "local":
    POSTS_PATH = PROJECT_ROOT / LOCAL_POSTS_DIR
    OUTPUT_DIR = LOCAL_WORK_DIR / LOCAL_OUTPUT_DIR
    POSTS_SPARK_PATH = str(POSTS_PATH)
    SPARK_BASE_PATH = str(POSTS_PATH)
    MONTHLY_PARQUET_SPARK = str(OUTPUT_DIR / "monthly_A")
elif DATA_SOURCE == "s3":
    POSTS_PATH = S3_POSTS_PATH.rstrip("/")
    OUTPUT_DIR = LOCAL_WORK_DIR / LOCAL_OUTPUT_DIR
    POSTS_SPARK_PATH = POSTS_PATH
    SPARK_BASE_PATH = POSTS_PATH
    MONTHLY_PARQUET_SPARK = f"{S3_OUTPUT_DIR.rstrip('/')}/monthly_A"
else:
    raise ValueError("DATA_SOURCE must be 'local' or 's3'")

MONTHLY_PARQUET = OUTPUT_DIR / "monthly_A"
MONTHLY_CSV_DIR = OUTPUT_DIR / "monthly_A_csv"
MONTHLY_FLAT_CSV = OUTPUT_DIR / "monthly_A_flat.csv"
RDD_RESULTS_CSV = OUTPUT_DIR / "rdd_results_A.csv"
CUTOFF = pd.Timestamp(CUTOFF_DATE)

OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data source: {DATA_SOURCE}")
print(f"Posts path: {POSTS_PATH}")
if DATA_SOURCE == "local":
    print(f"Posts path exists: {POSTS_PATH.exists()}")
print(f"Spark input path: {POSTS_SPARK_PATH}")
print(f"Spark monthly parquet output: {MONTHLY_PARQUET_SPARK}")
print(f"Local output dir: {OUTPUT_DIR}")
print(f"Sample month: {SAMPLE_MONTH}")
print(f"RDD cutoff: {CUTOFF.date()}")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Project root: /
Data source: s3
Posts path: s3://macs30123-final-pj-yanjing-li-2026/posts_cqi_with_tier
Spark input path: s3://macs30123-final-pj-yanjing-li-2026/posts_cqi_with_tier
Spark monthly parquet output: s3://macs30123-final-pj-yanjing-li-2026/outputs/monthly_A
Local output dir: /var/lib/livy/outputs
Sample month: 2022-11
RDD cutoff: 2022-12-01

## Small-Scale Spark Aggregation Test

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, count

spark = SparkSession.builder.appName("python_cqi_rdd_analysis_sample").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("WARN")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", str(max(sc.defaultParallelism * 2, 16)))

SAMPLE_SPARK_PATH = f"{POSTS_SPARK_PATH.rstrip('/')}/year_month={SAMPLE_MONTH}"

sample_df = spark.read.option("basePath", SPARK_BASE_PATH).parquet(SAMPLE_SPARK_PATH)
print(f"Sample path: {SAMPLE_SPARK_PATH}")
print("Spark version:", spark.version)
print("Default parallelism:", sc.defaultParallelism)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print(f"Sample rows: {sample_df.count():,}")
sample_df.printSchema()

sample_df = (
    sample_df.withColumn("cqi", col("cqi_version_a").cast("double"))
      .withColumn("score_num", col("Score").cast("double"))
      .withColumn("answer_count_num", col("AnswerCount").cast("double"))
      .filter(col("cqi").isNotNull())
      .filter(col("experience_tier").isNotNull())
)

sample_df = sample_df.withColumn(
    "tag_category",
    when(col("Tags").rlike("pandas|numpy|scipy|matplotlib|seaborn|dataframe"), "data_science")
    .when(col("Tags").rlike("tensorflow|pytorch|keras|machine-learning|deep-learning"), "ml_ai")
    .when(col("Tags").rlike("django|flask|fastapi|web|api"), "web")
    .when(col("Tags").rlike("selenium|scrapy|beautifulsoup|automation"), "automation")
    .when(col("Tags").rlike("python-3.x|python|list|dictionary|string|oop"), "core_python")
    .otherwise("other")
)

sample_monthly = sample_df.groupBy("year_month", "experience_tier", "tag_category").agg(
    avg("cqi").alias("mean_CQI"),
    count("*").alias("n_posts"),
    avg("score_num").alias("mean_score"),
    avg("answer_count_num").alias("mean_answer_count"),
)

sample_monthly.orderBy("experience_tier", "tag_category").show(50, truncate=False)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Sample path: s3://macs30123-final-pj-yanjing-li-2026/posts_cqi_with_tier/year_month=2022-11
Spark version: 3.0.1-amzn-0
Default parallelism: 4
Shuffle partitions: 16
Sample rows: 19,854
root
 |-- Id: string (nullable = true)
 |-- PostTypeId: string (nullable = true)
 |-- CreationDate: string (nullable = true)
 |-- Score: string (nullable = true)
 |-- Tags: string (nullable = true)
 |-- OwnerUserId: string (nullable = true)
 |-- AnswerCount: string (nullable = true)
 |-- ClosedDate: string (nullable = true)
 |-- AcceptedAnswerId: string (nullable = true)
 |-- is_python: integer (nullable = true)
 |-- is_duplicate: integer (nullable = true)
 |-- code_block_count: integer (nullable = true)
 |-- primary_code_len: integer (nullable = true)
 |-- cyclomatic_complexity: float (nullable = true)
 |-- halstead_volume: float (nullable = true)
 |-- is_parseable: float (nullable = true)
 |-- has_code: integer (nullable = true)
 |-- cqi_version_a: double (nullable = true)
 |-- Reputation: integer (nu

## Smoke Test


In [35]:
import statsmodels.api as sm

# Medium-small smoke test: a narrow window around the ChatGPT cutoff.
SMOKE_START = pd.Timestamp("2021-12-01")
SMOKE_END = pd.Timestamp("2023-12-01")
SMOKE_MONTHS = pd.date_range(SMOKE_START, SMOKE_END, freq="MS").strftime("%Y-%m").tolist()
SMOKE_PATHS = [f"{POSTS_SPARK_PATH.rstrip('/')}/year_month={m}" for m in SMOKE_MONTHS]

smoke_df = spark.read.option("basePath", SPARK_BASE_PATH).parquet(*SMOKE_PATHS)
print(f"Smoke-test months: {SMOKE_MONTHS[0]} to {SMOKE_MONTHS[-1]} ({len(SMOKE_MONTHS)} months)")
print(f"Smoke-test rows: {smoke_df.count():,}")

smoke_df = (
    smoke_df.withColumn("cqi", col("cqi_version_a").cast("double"))
      .withColumn("score_num", col("Score").cast("double"))
      .withColumn("answer_count_num", col("AnswerCount").cast("double"))
      .filter(col("cqi").isNotNull())
      .filter(col("experience_tier").isNotNull())
)

smoke_df = smoke_df.withColumn(
    "tag_category",
    when(col("Tags").rlike("pandas|numpy|scipy|matplotlib|seaborn|dataframe"), "data_science")
    .when(col("Tags").rlike("tensorflow|pytorch|keras|machine-learning|deep-learning"), "ml_ai")
    .when(col("Tags").rlike("django|flask|fastapi|web|api"), "web")
    .when(col("Tags").rlike("selenium|scrapy|beautifulsoup|automation"), "automation")
    .when(col("Tags").rlike("python-3.x|python|list|dictionary|string|oop"), "core_python")
    .otherwise("other")
)

smoke_monthly_sp = smoke_df.groupBy("year_month", "experience_tier", "tag_category").agg(
    avg("cqi").alias("mean_CQI"),
    count("*").alias("n_posts"),
    avg("score_num").alias("mean_score"),
    avg("answer_count_num").alias("mean_answer_count"),
)

smoke_monthly = smoke_monthly_sp.orderBy("year_month", "experience_tier", "tag_category").toPandas()
smoke_monthly["month"] = pd.to_datetime(smoke_monthly["year_month"])
show_table(smoke_monthly)

smoke_overall = pd.DataFrame([
    {
        "month": month,
        "mean_CQI": np.average(g["mean_CQI"], weights=g["n_posts"]),
        "n_posts": g["n_posts"].sum(),
    }
    for month, g in smoke_monthly.groupby("month")
]).sort_values("month")

plt.figure(figsize=(10, 4))
plt.plot(smoke_overall["month"], smoke_overall["mean_CQI"], linewidth=2)
plt.axvline(CUTOFF, color="red", linestyle="--", label="ChatGPT release")
plt.title("Smoke Test: Monthly Mean CQI Around Cutoff")
plt.xlabel("Month")
plt.ylabel("Weighted Mean CQI")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "smoke_cqi_trend.png", dpi=200)
plt.show()

# Debug one RDD regression on the smoke window. This checks code shape, not final inference.
smoke_sample = smoke_monthly[
    (smoke_monthly["experience_tier"] == DEBUG_TIER) &
    (smoke_monthly["tag_category"] == DEBUG_TAG_CATEGORY)
].copy()

if len(smoke_sample) >= 12 and smoke_sample["month"].lt(CUTOFF).any() and smoke_sample["month"].ge(CUTOFF).any():
    smoke_sample = smoke_sample.sort_values("month")
    smoke_sample["t"] = (smoke_sample["month"] - CUTOFF).dt.days / 30
    smoke_sample["post"] = (smoke_sample["month"] >= CUTOFF).astype(int)
    smoke_sample["post_t"] = smoke_sample["post"] * smoke_sample["t"]
    smoke_X = sm.add_constant(smoke_sample[["post", "t", "post_t"]])
    smoke_model = sm.OLS(smoke_sample["mean_CQI"], smoke_X).fit()
    print(f"Smoke RDD: {DEBUG_TIER} x {DEBUG_TAG_CATEGORY}")
    print(smoke_model.summary())
else:
    print("Smoke RDD skipped: not enough pre/post months for the debug group.")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Smoke-test months: 2021-12 to 2023-12 (25 months)
Smoke-test rows: 377,510
year_month experience_tier  tag_category  mean_CQI  n_posts  mean_score  mean_answer_count      month
   2021-12          expert    automation  0.547044       27    0.629630           1.111111 2021-12-01
   2021-12          expert   core_python  0.723108     1372    1.549563           1.223761 2021-12-01
   2021-12          expert  data_science  0.719953      632    0.995253           1.324367 2021-12-01
   2021-12          expert         ml_ai  0.792795      127    1.755906           1.094488 2021-12-01
   2021-12          expert           web  0.715505      279    2.448029           1.086022 2021-12-01
   2021-12    intermediate    automation  0.704067      104    0.615385           1.307692 2021-12-01
   2021-12    intermediate   core_python  0.740653     2863    0.948306           1.206078 2021-12-01
   2021-12    intermediate  data_science  0.693340     1525    0.929180           1.353443 2021-12-01
   2021

## Full-Scale Spark Aggregation

In [23]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, count

spark = SparkSession.builder.appName("python_cqi_rdd_analysis_full").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("WARN")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", str(max(sc.defaultParallelism * 2, 16)))

df = spark.read.option("basePath", SPARK_BASE_PATH).parquet(POSTS_SPARK_PATH)

print("Input schema:")
df.printSchema()

df = (
    df.withColumn("cqi", col("cqi_version_a").cast("double"))
      .withColumn("score_num", col("Score").cast("double"))
      .withColumn("answer_count_num", col("AnswerCount").cast("double"))
      .filter(col("cqi").isNotNull())
      .filter(col("experience_tier").isNotNull())
)

df = df.withColumn(
    "tag_category",
    when(col("Tags").rlike("pandas|numpy|scipy|matplotlib|seaborn|dataframe"), "data_science")
    .when(col("Tags").rlike("tensorflow|pytorch|keras|machine-learning|deep-learning"), "ml_ai")
    .when(col("Tags").rlike("django|flask|fastapi|web|api"), "web")
    .when(col("Tags").rlike("selenium|scrapy|beautifulsoup|automation"), "automation")
    .when(col("Tags").rlike("python-3.x|python|list|dictionary|string|oop"), "core_python")
    .otherwise("other")
)

monthly_A = df.groupBy("year_month", "experience_tier", "tag_category").agg(
    avg("cqi").alias("mean_CQI"),
    count("*").alias("n_posts"),
    avg("score_num").alias("mean_score"),
    avg("answer_count_num").alias("mean_answer_count"),
)

monthly_A.write.mode("overwrite").parquet(MONTHLY_PARQUET_SPARK)

# The monthly table is small, so bring it to the notebook driver for statsmodels and plotting.
monthly_A_pd = monthly_A.orderBy("year_month", "experience_tier", "tag_category").toPandas()
monthly_A_pd.to_csv(MONTHLY_FLAT_CSV, index=False)

print(f"Wrote monthly parquet to {MONTHLY_PARQUET_SPARK}")
print(f"Wrote local flat monthly CSV to {MONTHLY_FLAT_CSV}")
show_table(monthly_A_pd)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Input schema:
root
 |-- Id: string (nullable = true)
 |-- PostTypeId: string (nullable = true)
 |-- CreationDate: string (nullable = true)
 |-- Score: string (nullable = true)
 |-- Tags: string (nullable = true)
 |-- OwnerUserId: string (nullable = true)
 |-- AnswerCount: string (nullable = true)
 |-- ClosedDate: string (nullable = true)
 |-- AcceptedAnswerId: string (nullable = true)
 |-- is_python: integer (nullable = true)
 |-- is_duplicate: integer (nullable = true)
 |-- code_block_count: integer (nullable = true)
 |-- primary_code_len: integer (nullable = true)
 |-- cyclomatic_complexity: float (nullable = true)
 |-- halstead_volume: float (nullable = true)
 |-- is_parseable: float (nullable = true)
 |-- has_code: integer (nullable = true)
 |-- cqi_version_a: double (nullable = true)
 |-- Reputation: integer (nullable = true)
 |-- experience_tier: string (nullable = true)
 |-- year_month: string (nullable = true)

Wrote monthly parquet to s3://macs30123-final-pj-yanjing-li-2026/ou

## 2. Load and Inspect Monthly Table

In [24]:
if MONTHLY_FLAT_CSV.exists():
    monthly = pd.read_csv(MONTHLY_FLAT_CSV)
else:
    csv_files = sorted(glob.glob(str(MONTHLY_CSV_DIR / "*.csv")))
    assert csv_files, f"No monthly CSV files found in {MONTHLY_CSV_DIR} or {MONTHLY_FLAT_CSV}"
    monthly = pd.read_csv(csv_files[0])

monthly["month"] = pd.to_datetime(monthly["year_month"])
monthly = monthly.sort_values(["month", "experience_tier", "tag_category"]).reset_index(drop=True)

show_table(monthly_A_pd)
print(f"Rows: {len(monthly):,}")
print(f"Month range: {monthly['year_month'].min()} to {monthly['year_month'].max()}")
print("\nExperience tiers:")
print(monthly["experience_tier"].value_counts())
print("\nTag categories:")
print(monthly["tag_category"].value_counts())
print("\nMissing values:")
print(monthly.isna().sum())


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

year_month experience_tier  tag_category  mean_CQI  n_posts  mean_score  mean_answer_count
   2008-08          expert   core_python  1.475202      101  202.881188           9.663366
   2008-08          expert           web  1.406779       14   72.857143           6.785714
   2008-08    intermediate   core_python  0.983590        1    9.000000           7.000000
   2008-08          novice   core_python  0.865653        1    6.000000           5.000000
   2008-09          expert    automation  1.113713        4   44.500000           7.750000
   2008-09          expert   core_python  1.358447      391  253.933504           9.202046
   2008-09          expert  data_science  1.364946        4   83.750000           4.750000
   2008-09          expert           web  1.275320       48   61.020833           6.395833
   2008-09    intermediate   core_python  1.007745       45   78.955556           5.444444
   2008-09    intermediate           web  1.410251        4   20.750000           4.250000

## 3. Overall CQI Trend

The overall mean is weighted by `n_posts`, so larger monthly groups contribute proportionally.

In [44]:
import matplotlib.pyplot as plt
import pandas as pd

overall = pd.DataFrame([
    {
        "month": month,
        "mean_CQI": np.average(g["mean_CQI"], weights=g["n_posts"]),
        "n_posts": g["n_posts"].sum(),
    }
    for month, g in monthly.groupby("month")
]).sort_values("month")

plt.figure(figsize=(12, 5))
plt.plot(overall["month"], overall["mean_CQI"], linewidth=2)
plt.axvline(CUTOFF, color="red", linestyle="--", label="ChatGPT release")
plt.title("Monthly Mean CQI Over Time")
plt.xlabel("Month")
plt.ylabel("Weighted Mean CQI")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "overall_cqi_trend.png", dpi=200)

show_table(overall.tail())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

     month  mean_CQI  n_posts
2024-11-01  0.729241     3278
2024-12-01  0.752532     2808
2025-01-01  0.755632     2797
2025-02-01  0.734687     2335
2025-03-01  0.704770     2795

## 4. Trends by Experience Tier

In [46]:
import seaborn as sns

tier_trend = pd.DataFrame([
    {
        "month": month,
        "experience_tier": tier,
        "mean_CQI": np.average(g["mean_CQI"], weights=g["n_posts"]),
        "n_posts": g["n_posts"].sum(),
    }
    for (month, tier), g in monthly.groupby(["month", "experience_tier"])
]).sort_values(["month", "experience_tier"])

plt.figure(figsize=(12, 5))
sns.lineplot(data=tier_trend, x="month", y="mean_CQI", hue="experience_tier", linewidth=2)
plt.axvline(CUTOFF, color="red", linestyle="--", label="ChatGPT release")
plt.title("Monthly Mean CQI by Experience Tier")
plt.xlabel("Month")
plt.ylabel("Weighted Mean CQI")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cqi_trend_by_tier.png", dpi=200)
plt.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## 5. Single-Group RDD Debug

Start with one group to verify the model before running all groups.

In [47]:
def prepare_rdd_frame(g, cutoff=CUTOFF):
    g = g.copy()
    g["month"] = pd.to_datetime(g["year_month"])
    g = g.sort_values("month")
    g["t"] = (g["month"] - cutoff).dt.days / 30
    g["post"] = (g["month"] >= cutoff).astype(int)
    g["post_t"] = g["post"] * g["t"]
    return g

def fit_rdd(g):
    g = prepare_rdd_frame(g)
    X = sm.add_constant(g[["post", "t", "post_t"]])
    model = sm.OLS(g["mean_CQI"], X).fit()
    return g, X, model

sample_tier = DEBUG_TIER
sample_category = DEBUG_TAG_CATEGORY

sample = monthly[
    (monthly["experience_tier"] == sample_tier) &
    (monthly["tag_category"] == sample_category)
].copy()

sample_rdd, sample_X, sample_model = fit_rdd(sample)
print(sample_model.summary())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

                            OLS Regression Results                            
Dep. Variable:               mean_CQI   R-squared:                       0.404
Model:                            OLS   Adj. R-squared:                  0.394
Method:                 Least Squares   F-statistic:                     41.74
Date:                Mon, 25 May 2026   Prob (F-statistic):           1.19e-20
Time:                        22:38:25   Log-Likelihood:                 263.82
No. Observations:                 189   AIC:                            -519.6
Df Residuals:                     185   BIC:                            -506.7
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.6103      0.010     63.763      0.0

In [48]:
sample_rdd["fitted"] = sample_model.predict(sample_X)

plt.figure(figsize=(10, 5))
plt.scatter(sample_rdd["month"], sample_rdd["mean_CQI"], s=30, label="Observed")
plt.plot(sample_rdd["month"], sample_rdd["fitted"], color="black", linewidth=2, label="RDD fitted")
plt.axvline(CUTOFF, color="red", linestyle="--", label="ChatGPT release")
plt.title(f"RDD Debug: {sample_tier} x {sample_category}")
plt.xlabel("Month")
plt.ylabel("Mean CQI")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"rdd_{sample_tier}_{sample_category}.png", dpi=200)
plt.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## 6. Run RDD for Every Group

`beta_post` is the immediate discontinuity at the cutoff. Negative values mean mean CQI fell at the ChatGPT release cutoff.

In [49]:
def run_group_rdd(g, min_months=24):
    g = prepare_rdd_frame(g)
    if len(g) < min_months:
        return None
    if g["post"].nunique() < 2:
        return None

    X = sm.add_constant(g[["post", "t", "post_t"]])
    model = sm.OLS(g["mean_CQI"], X).fit()
    ci = model.conf_int().loc["post"]

    return {
        "beta_post": model.params["post"],
        "pvalue": model.pvalues["post"],
        "ci_low": ci[0],
        "ci_high": ci[1],
        "n_months": len(g),
        "n_posts": g["n_posts"].sum(),
        "pre_months": int((g["post"] == 0).sum()),
        "post_months": int((g["post"] == 1).sum()),
    }

results = []
for (tier, category), g in monthly.groupby(["experience_tier", "tag_category"]):
    row = run_group_rdd(g)
    if row is None:
        continue
    row["experience_tier"] = tier
    row["tag_category"] = category
    results.append(row)

rdd_results = pd.DataFrame(results)
rdd_results = rdd_results[
    ["experience_tier", "tag_category", "beta_post", "pvalue", "ci_low", "ci_high", "n_months", "pre_months", "post_months", "n_posts"]
].sort_values("beta_post")

rdd_results.to_csv(RDD_RESULTS_CSV, index=False)
show_table(rdd_results, n=len(rdd_results))
print(f"Saved RDD results to {RDD_RESULTS_CSV}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

experience_tier  tag_category  beta_post    pvalue    ci_low   ci_high  n_months  pre_months  post_months  n_posts
         expert    automation  -0.036833  0.510349 -0.146986  0.073321       196         170           26     8996
   intermediate         ml_ai  -0.011687  0.820952 -0.113473  0.090099       172         144           28    28649
         novice         ml_ai  -0.010809  0.863175 -0.134511  0.112893       157         129           28    40082
         novice   core_python   0.011914  0.395537 -0.015680  0.039507       200         172           28   666592
   intermediate   core_python   0.011967  0.541858 -0.026655  0.050590       200         172           28   413979
   intermediate    automation   0.014362  0.798085 -0.096241  0.124966       188         160           28    13707
         expert         ml_ai   0.018444  0.764219 -0.102719  0.139606       181         153           28    19632
         novice           web   0.033114  0.091240 -0.005365  0.071593       196

## 7. Result Summaries

In [50]:
tier_summary = (
    rdd_results.groupby("experience_tier", as_index=False)
    .agg(
        mean_beta_post=("beta_post", "mean"),
        median_beta_post=("beta_post", "median"),
        significant_groups=("pvalue", lambda s: int((s < 0.05).sum())),
        groups=("tag_category", "count"),
    )
    .sort_values("mean_beta_post")
)

category_summary = (
    rdd_results.groupby("tag_category", as_index=False)
    .agg(
        mean_beta_post=("beta_post", "mean"),
        median_beta_post=("beta_post", "median"),
        significant_groups=("pvalue", lambda s: int((s < 0.05).sum())),
        groups=("experience_tier", "count"),
    )
    .sort_values("mean_beta_post")
)

show_table(tier_summary, n=len(tier_summary))
show_table(category_summary, n=len(category_summary))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

mean_beta_post  median_beta_post  significant_groups  groups
        expert          0.038282            0.050840     2.0
  intermediate          0.034983            0.014362     2.0
        novice          0.036267            0.033114     1.0
mean_beta_post  median_beta_post  significant_groups  groups
    automation          0.029427            0.014362     1.0
   core_python          0.024907            0.011967     0.0
  data_science          0.078690            0.094765     2.0
         ml_ai         -0.001351           -0.010809     0.0
           web          0.050880            0.055329     2.0

## 8. Beta Heatmap

In [51]:
tier_order = ["novice", "intermediate", "expert"]
heat = rdd_results.pivot(index="experience_tier", columns="tag_category", values="beta_post")
heat = heat.reindex([tier for tier in tier_order if tier in heat.index])

plt.figure(figsize=(10, 4.5))
sns.heatmap(heat, annot=True, fmt=".3f", cmap="RdBu_r", center=0, linewidths=0.5)
plt.title("RDD Effect of ChatGPT Release on CQI")
plt.xlabel("Tag Category")
plt.ylabel("Experience Tier")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "beta_heatmap.png", dpi=200)
plt.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## 9. Confidence Intervals

In [52]:
plot_df = rdd_results.sort_values("beta_post").reset_index(drop=True)
labels = plot_df["experience_tier"] + " | " + plot_df["tag_category"]

plt.figure(figsize=(9, max(5, 0.35 * len(plot_df))))
plt.errorbar(
    plot_df["beta_post"],
    np.arange(len(plot_df)),
    xerr=[plot_df["beta_post"] - plot_df["ci_low"], plot_df["ci_high"] - plot_df["beta_post"]],
    fmt="o",
    color="black",
    ecolor="gray",
    capsize=3,
)
plt.axvline(0, color="red", linestyle="--", linewidth=1)
plt.yticks(np.arange(len(plot_df)), labels)
plt.xlabel("Beta Post with 95% CI")
plt.title("RDD Estimates by Group")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "rdd_confidence_intervals.png", dpi=200)
plt.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## 10. Clean Tables for Dashboard or Report

In [53]:
monthly.to_csv(OUTPUT_DIR / "monthly_A_flat.csv", index=False)
tier_summary.to_csv(OUTPUT_DIR / "tier_beta_summary.csv", index=False)
category_summary.to_csv(OUTPUT_DIR / "category_beta_summary.csv", index=False)

print("Exported:")
for path in [
    OUTPUT_DIR / "monthly_A_flat.csv",
    OUTPUT_DIR / "rdd_results_A.csv",
    OUTPUT_DIR / "tier_beta_summary.csv",
    OUTPUT_DIR / "category_beta_summary.csv",
    OUTPUT_DIR / "overall_cqi_trend.png",
    OUTPUT_DIR / "cqi_trend_by_tier.png",
    OUTPUT_DIR / "beta_heatmap.png",
    OUTPUT_DIR / "rdd_confidence_intervals.png",
]:
    print(path)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Exported:
/var/lib/livy/outputs/monthly_A_flat.csv
/var/lib/livy/outputs/rdd_results_A.csv
/var/lib/livy/outputs/tier_beta_summary.csv
/var/lib/livy/outputs/category_beta_summary.csv
/var/lib/livy/outputs/overall_cqi_trend.png
/var/lib/livy/outputs/cqi_trend_by_tier.png
/var/lib/livy/outputs/beta_heatmap.png
/var/lib/livy/outputs/rdd_confidence_intervals.png

## 11. Upload Outputs to S3

Run this at the end on EMR to preserve result tables and figures in your project bucket.


In [69]:
# Upload local notebook outputs back to S3.
if DATA_SOURCE == "s3":
    try:
        import subprocess
        subprocess.run(["aws", "s3", "sync", str(OUTPUT_DIR), S3_OUTPUT_DIR.rstrip("/") + "/"], check=True)
        print(f"Uploaded {OUTPUT_DIR} to {S3_OUTPUT_DIR}")
    except Exception as exc:
        print("AWS CLI upload failed. You can upload manually with:")
        print(f"aws s3 sync {OUTPUT_DIR} {S3_OUTPUT_DIR.rstrip('/')}/")
        print(exc)
else:
    print("DATA_SOURCE is local; skipping S3 upload.")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

CompletedProcess(args=['aws', 's3', 'sync', '/var/lib/livy/outputs', 's3://macs30123-final-pj-yanjing-li-2026/outputs/'], returncode=0)
Uploaded /var/lib/livy/outputs to s3://macs30123-final-pj-yanjing-li-2026/outputs

In [70]:
spark.stop()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Uploaded /var/lib/livy/outputs to s3://macs30123-final-pj-yanjing-li-2026/outputs